In [ ]:
import numpy as np
from numpy import ndarray
import matplotlib.pyplot as plt


np.__version__

### Реализация функций активации.

In [ ]:
# ReLU
def relu(x: ndarray) -> ndarray:
    return np.maximum(0, x)

def drelu(act: ndarray, grad: ndarray) -> ndarray:
    return grad*np.where(act>0, 1, 0)


In [ ]:
def func_act(x: ndarray, f_act) -> ndarray:
    """Прямой проход через функцию активации."""
    return f_act(x)


def func_grad(act: ndarray, grad: ndarray, f_grad) -> ndarray:
    """
    Обратный проход через функцию активации.
    """
    return f_grad(act, grad)


def initialization_param(
        size: tuple[int, int],
        mode: str = "randn",
        fan: str = "fan_in",
        seed: int = None
) -> ndarray:
    """
    Инициализация параметров сети.
    """
    rng = np.random.default_rng(seed)
    fan_in, fan_out = size
    gain = np.sqrt(2.0)

    if mode == "randn":
        return rng.standard_normal(size=size)

    elif mode == "xavier_uniform":
        std = gain * np.sqrt(2.0 / (fan_in + fan_out))
        bound = np.sqrt(3.0) * std 
        return rng.uniform(-bound, bound, size=size)

    elif mode == "xavier_normal":
        std = gain * np.sqrt(2.0 / (fan_in + fan_out))
        return rng.normal(0, std, size=size)
    
    elif mode == "kaiming_uniform":
        std = gain / np.sqrt(fan_in if fan == "fan_in" else fan_out)
        bound = np.sqrt(3.0) * std 
        return rng.uniform(-bound, bound, size=size)

    elif mode == "kaiming_normal":
        std = gain / np.sqrt(fan_in if fan == "fan_in" else fan_out)
        return rng.normal(0, std, size=size)

    else:
        raise ValueError(
            f"Параметр 'mode' принимает значения: "
            f"'randn', 'xavier_uniform', 'xavier_normal', 'kaiming_uniform', 'kaiming_normal'. "
            f"Вы передали '{mode}'."
        )

In [ ]:
def plot(
        name: str,
        init_mode: str,
        fan: str,
        data: dict
) -> None:
    """
    Визуализация прямого и обратного прохода.
    """

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    fig.suptitle(f'{name}, {init_mode}, {fan} ', fontsize=14, fontweight='bold')

    for ax, (mode, bef_aft) in zip(axes, data.items()):

        n = len(bef_aft['до активации'])
        labels = [f'Layer_{j + 1}' for j in range(n)]
        x_pos = np.arange(n)
        width = 0.35

        before = bef_aft['до активации']
        after  = bef_aft['после активации']

        if mode == 'forward':
            ax.bar(x_pos - width / 2, before, width,
                   label=f'до {name}', color='steelblue')
            ax.bar(x_pos + width / 2, after, width,
                   label=f'после {name}', color='tomato')
            ax.set_title('Forward Pass — значение std', fontsize=12)
            ax.set_ylabel('std')

        else:  # backward
            ax.bar(x_pos - width / 2, after, width,
                   label=f'после {name}', color='steelblue')
            ax.bar(x_pos + width / 2, before, width,
                   label=f'до {name}', color='tomato')
            ax.set_title('Backward Pass — значение std', fontsize=12)
            ax.set_ylabel('std')

        ax.set_xlabel('Слой')
        ax.set_xticks(x_pos)
        ax.set_xticklabels(labels)
        ax.legend()
        ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.show()

def func(activation_configs, seed):
    layer_dims = [(256, 128), (128, 64), (64, 32), (32, 16), (16, 8)]

    rng = np.random.default_rng(seed)
    x = rng.standard_normal((16, 256))

    for config in activation_configs:
        name = config['name']
        init_mode = config['init_mode']
        fan = config.get('fan')
        f_act = config['f_act']
        f_grad = config['f_grad']

        # Инициализируем параметры сети
        weights = []
        biases  = []
        for j, (n_in, n_out) in enumerate(layer_dims):
            W = initialization_param((n_in, n_out), mode=init_mode, fan=fan, seed=seed + j)
            b = np.zeros((1, n_out))  # bias = 0
            weights.append(W)
            biases.append(b)

        activations     = []   # выходы активаций

        before_act_std  = []
        after_act_std   = []

        current = x
        for W, b in zip(weights, biases):
            pre = current @ W + b
            act = func_act(pre, f_act)

            activations.append(act)

            before_act_std.append(pre.std())
            after_act_std.append(act.std())

            current = act

        before_grad_std = []
        after_grad_std  = []

        # градиент от loss ~ N(0, 1)
        grad = rng.standard_normal(activations[-1].shape)

        for layer_idx in reversed(range(len(layer_dims))):

            before_grad_std.append(grad.std())

            grad = func_grad(activations[layer_idx], grad, f_grad)

            after_grad_std.append(grad.std())

            if layer_idx > 0:
                grad = grad @ weights[layer_idx].T

        before_grad_std = before_grad_std[::-1]
        after_grad_std  = after_grad_std[::-1]

        data = {
            'forward': {
                'до активации':    before_act_std,
                'после активации': after_act_std,
            },
            'backward': {
                'до активации':    before_grad_std,
                'после активации': after_grad_std,
            },
        }

        plot(name, init_mode, fan, data)

In [ ]:
seed = 1961

#'randn', 'xavier_uniform', 'xavier_normal', 'kaiming_uniform', 'kaiming_normal'. "
act_name = 'ReLU'
act = relu
grad = drelu

activation_configs = [
    {
        'name': act_name,
        'init_mode': 'randn',
        'f_act':  act,
        'f_grad': grad,
    },
    {
        'name': act_name,
        'init_mode': 'xavier_normal',
        'f_act':  act,
        'f_grad': grad,
    },
    {
        'name': act_name,
        'init_mode': 'kaiming_uniform',
        'fan': "fan_out",
        'f_act':  act,
        'f_grad': grad,
    },
    {
        'name': act_name,
        'init_mode': 'kaiming_uniform',
        'fan': "fan_in",
        'f_act':  act,
        'f_grad': grad,
    },
    {
        'name': act_name,
        'init_mode': 'kaiming_normal',
        'fan': "fan_out",
        'f_act':  act,
        'f_grad': grad,
    },
    {
        'name': act_name,
        'init_mode': 'kaiming_normal',
        'fan': "fan_in",
        'f_act':  act,
        'f_grad': grad,
    },
]

func(activation_configs, seed)